# Multilayer Sweep Comparison

Compare reflectance spectra across a set of runs that differ in the number of
chitin/air bilayers (or any other single swept parameter).

**Workflow**
1. Run the simulation in `main.py` once per configuration, giving each run a
   descriptive name (e.g. `menelaus_2layers`, `menelaus_6layers`, …).
2. Fill in the `RUNS` list below and run all cells.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np


In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
RESULTS_DIR  = Path("../results")
WL_MIN       = 380       # nm
WL_MAX       = 780       # nm
SAVE_FIGURE  = False     # True → save the figure into the first run's folder

# Species whose runs you want to compare
SPECIES = "menelaus"

# List of (run_name, legend_label) tuples — one entry per simulation run.
# run_name must match the experiment name used in main.py.
# label     is what appears in the legend (e.g. the number of layers).
RUNS = [
    ("menelaus_2layers",   "2 layers"),
    ("menelaus_4layers",   "4 layers"),
    ("menelaus_6layers",   "6 layers"),
    ("menelaus_8layers",   "8 layers"),
    ("menelaus_10layers",  "10 layers"),
    ("menelaus_12layers",  "12 layers"),
]

TAG = "total"   # which result tag to load from each run

# ── Verify which runs are actually present ────────────────────────────────
print("Run availability:")
for run_name, label in RUNS:
    path = RESULTS_DIR / SPECIES / run_name / f"{run_name}_{TAG}.csv"
    status = "✓ found" if path.exists() else "✗ missing"
    print(f"  {status}  {run_name}  ({label})")


In [ ]:
# Load available datasets
datasets = {}   # label → DataFrame
for run_name, label in RUNS:
    path = RESULTS_DIR / SPECIES / run_name / f"{run_name}_{TAG}.csv"
    if path.exists():
        datasets[label] = pd.read_csv(path).sort_values("wavelength_nm")
    else:
        print(f"[SKIP] {run_name}: file not found — run the simulation first.")

print(f"\nLoaded {len(datasets)} / {len(RUNS)} runs: {list(datasets.keys())}")


In [ ]:
if not datasets:
    print("[INFO] No data loaded — run at least one simulation first.")
else:
    n      = len(datasets)
    colors = cm.cool(np.linspace(0.1, 0.9, n))

    fig, ax = plt.subplots(figsize=(10, 6))

    for (label, df), color in zip(datasets.items(), colors):
        ax.plot(df["wavelength_nm"], df["reflectance"],
                label=label, color=color, linewidth=1.8)

        # Mark peak within the wavelength window
        win = (df["wavelength_nm"] >= WL_MIN) & (df["wavelength_nm"] <= WL_MAX)
        if win.any():
            idx_peak = df[win]["reflectance"].idxmax()
            peak_wl  = df.loc[idx_peak, "wavelength_nm"]
            ax.axvline(peak_wl, color=color, linestyle=":", linewidth=1, alpha=0.5)

    ax.set_xlabel("Wavelength (nm)", fontsize=13)
    ax.set_ylabel("Reflectance",     fontsize=13)
    ax.set_title(
        fr"$\it{{{SPECIES}}}$ — Reflectance vs. Number of Bilayers  [{TAG}]",
        fontsize=14
    )
    ax.legend(title="Configuration", fontsize=10, title_fontsize=11)
    ax.set_xlim(WL_MIN, WL_MAX)
    ax.set_ylim(bottom=0)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()

    if SAVE_FIGURE:
        first_run = RUNS[0][0]
        out = RESULTS_DIR / SPECIES / first_run / f"{SPECIES}_multilayer_comparison.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        print(f"[OK] Figure saved: {out}")

    plt.show()
